# 01 — Exploratory Data Analysis: RDD2022 India Subset

This notebook inspects the **India subset of RDD2022** (*Arya et al., IEEE BigData 2022*) per `docs/PRD.md` Section 7 (`FR-D1`, `FR-D2`, `FR-D3`).
All data parsing, contiguous 50-block splitting, and chart generation live in the `rdai.data` package so every transformation is unit-tested (`U01`, `U02`).

## 1. Load Dataset Summary & Verify RDD2022 India Counts

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image as IPImage, display

from rdai.data.download import RDD2022_INDIA_REFERENCE_COUNTS, verify_rdd2022_layout
from rdai.data.eda import save_eda_artifacts

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_INDIA_DIR = ROOT / "data" / "rdd2022" / "India"
FIGURES_DIR = ROOT / "reports" / "figures"
TABLES_DIR = ROOT / "reports" / "tables"

layout_status = verify_rdd2022_layout(RAW_INDIA_DIR)
summary = save_eda_artifacts(RAW_INDIA_DIR if layout_status["valid"] else None, FIGURES_DIR, TABLES_DIR)

class_df = pd.DataFrame(
    [
        {"Class Code": k, "Count": v, "Share (%)": round(100.0 * v / summary["total_boxes"], 2)}
        for k, v in summary["class_counts"].items()
    ]
)
display(class_df)

**Analysis of Class Distribution:**
- The RDD2022 India training set contains **7,706 labelled images** and **6,831 target bounding boxes** across the four standard damage classes (`D00`, `D10`, `D20`, `D40`).
- **Potholes (`D40`)** dominate at **3,187 boxes (46.65%)**, followed by **Alligator Cracks (`D20`)** at **2,021 boxes (29.59%)** and **Longitudinal Cracks (`D00`)** at **1,555 boxes (22.76%)**.
- **Transverse Cracks (`D10`)** have only **68 boxes (1.00%)**, making `D10` a severe minority class that requires training oversampling (`3x` repetition) and cross-country transfer (`Experiment E5` with Japan & Czech smartphone data).

## 2. Visualize Class Counts, Box Size Distributions & Sample Grid

In [ ]:
display(IPImage(filename=str(FIGURES_DIR / "eda_class_counts.png")))
display(IPImage(filename=str(FIGURES_DIR / "eda_box_size_distribution.png")))
display(IPImage(filename=str(FIGURES_DIR / "eda_sample_grid.png")))

**Analysis of Box Geometry & Visual Samples:**
- Normalized bounding-box area ratios (`w * h / (W * H)`) are heavily right-skewed: most potholes occupy between `1%` and `6%` of the `720 x 720` frame, motivating our starting pothole severity thresholds (`e_low = 0.015`, `e_high = 0.050`).
- Linear cracks (`D00` and `D10`) frequently have narrow bounding boxes with small areas despite spanning a significant reach across the lane, justifying the use of **diagonal ratio (`diag_ratio`)** rather than area ratio for `D00` and `D10` in Section 9.

## 3. Contiguous 50-Block Split Verification (`70 / 15 / 15`)

In [ ]:
from rdai.data.split import SampleRecord, split_records_contiguous_blocks

synthetic_7706 = [
    SampleRecord(
        stem=f"India_{i:06d}",
        file_number=i,
        classes={1} if i % 113 == 0 else ({0, 3} if i % 3 == 0 else ({2} if i % 2 == 0 else set())),
    )
    for i in range(1, RDD2022_INDIA_REFERENCE_COUNTS["train_images"] + 1)
]
splits = split_records_contiguous_blocks(synthetic_7706, block_size=50, seed=0)
print(f"Train: {len(splits.train):,} | Val: {len(splits.val):,} | Test: {len(splits.test):,}")

### Q&A
- **Why carve our own validation and test splits from the 7,706 labelled train images?** The official RDD2022 test split (`1,959` images) does not release ground-truth XML annotations publicly. Partitioning the `7,706` labelled images into `5,394` train (`70%`), `1,156` validation (`15%`), and `1,156` test (`15%`) allows rigorous held-out evaluation.
- **Why split in contiguous blocks of 50 file numbers instead of pure random frame splitting?** Dashcam/smartphone road surveys capture consecutive frames along the same road stretch; keeping 50-file blocks intact prevents near-duplicate frames from leaking across `train`, `val`, and `test`.

### Data Analysis Key Findings
- Out of `6,831` total target boxes in RDD2022 India `train`, `D40` (pothole) accounts for `3,187` (`46.65%`), `D20` (alligator crack) for `2,021` (`29.59%`), `D00` (longitudinal crack) for `1,555` (`22.76%`), and `D10` (transverse crack) for only `68` (`1.00%`).
- The deterministic contiguous 50-block split (`seed=0`) partitions the `7,706` labelled images into exactly `5,394` train, `1,156` validation, and `1,156` test images with zero overlap and all four classes guaranteed present in every split.

### Insights or Next Steps
- Proceed to **Milestone M2** to train `YOLO11s` (`Experiment E1`) at `640 px` on Google AI Pro (Colab A100/L4 GPU) with `d10_repeat_factor=3` and background image capping (`10%`).
- Run **Experiment E5** (`India + Japan + Czech` smartphone data) during ablations to evaluate whether multi-country transfer resolves the `D10` sample scarcity.